## Transformation and Claen up of the dataset 
In this section we are going to clean and tudy up the data after doing the exploration. We will drop some columns and encode a few so lets dive into it

In [86]:
#importing libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [87]:
#reading the dataset
raw_data = pd.read_csv("/Users/sa03/Desktop/Financial_Fraud_Detection/Dataset/raw_data.csv")   

#checking the shape of the dataset
raw_data.shape

(6362620, 11)

We can see that our dataset is over 6 million which is too large so we will generate a sample from the population and perform the rest of our analysis on.

In [88]:
#checking the information about the dataset

raw_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


In [89]:
# Lets create a stratified sample of 180000 rows from 
# the dataset to ensure that the distribution of the target variable is preserved in the sample.

from sklearn.model_selection import train_test_split
df, _ = train_test_split(raw_data, train_size=180000, stratify=raw_data['isFraud'], random_state=42)



In [90]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 180000 entries, 3175441 to 4541833
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   step            180000 non-null  int64  
 1   type            180000 non-null  object 
 2   amount          180000 non-null  float64
 3   nameOrig        180000 non-null  object 
 4   oldbalanceOrg   180000 non-null  float64
 5   newbalanceOrig  180000 non-null  float64
 6   nameDest        180000 non-null  object 
 7   oldbalanceDest  180000 non-null  float64
 8   newbalanceDest  180000 non-null  float64
 9   isFraud         180000 non-null  int64  
 10  isFlaggedFraud  180000 non-null  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 16.5+ MB


In [91]:
#Dropping the columns which are not required for the analysis

df.drop(columns=['nameOrig','nameDest','isFlaggedFraud'], inplace=True)

In [92]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 180000 entries, 3175441 to 4541833
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   step            180000 non-null  int64  
 1   type            180000 non-null  object 
 2   amount          180000 non-null  float64
 3   oldbalanceOrg   180000 non-null  float64
 4   newbalanceOrig  180000 non-null  float64
 5   oldbalanceDest  180000 non-null  float64
 6   newbalanceDest  180000 non-null  float64
 7   isFraud         180000 non-null  int64  
dtypes: float64(5), int64(2), object(1)
memory usage: 12.4+ MB


We are dropping all thses columns because they are not useful to our model and also something like nameOrig is just a random username and has no specfic or useful pattern in it

In [93]:
# We can see that the "type" column is of object data type. 

df['type']

3175441     CASH_IN
2283609    TRANSFER
4834889    CASH_OUT
6159828     PAYMENT
5962483    CASH_OUT
             ...   
5986750     PAYMENT
5498478    CASH_OUT
1663828    CASH_OUT
2155209    CASH_OUT
4541833    CASH_OUT
Name: type, Length: 180000, dtype: object

In [94]:
#lets hot encode the "type" column to convert it into numerical format

df = pd.get_dummies(df, columns=['type'], drop_first=True)

In [95]:
# Let us check the first 5 rows of the dataset after encoding

df.head()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
3175441,238,310473.41,4587838.10,4898311.51,2246945.59,1936472.18,0,False,False,False,False
2283609,187,669728.01,0.00,0.00,725008.39,1394736.40,0,False,False,False,True
4834889,347,18365.12,84182.18,65817.05,1799237.44,1817602.56,0,True,False,False,False
6159828,548,9752.53,10401.00,648.47,0.00,0.00,0,False,False,True,False
5962483,406,116752.19,448684.00,331931.81,0.00,116752.19,0,True,False,False,False


## Feature engineer a few fraud signals 

In [96]:
# How much did our original balance change

df['balance_change'] = df['oldbalanceOrg'] - df['newbalanceOrig']

print(df['balance_change'].head(10))

3175441   -310473.41
2283609         0.00
4834889     18365.13
6159828      9752.53
5962483    116752.19
3833646   -238230.29
2808571         0.00
3299225         0.00
4421469         0.00
529022       6308.47
Name: balance_change, dtype: float64


In [97]:
# How much did the destination balance change

df['dest_balance_change'] = df['oldbalanceDest'] - df['newbalanceDest']

print(df['dest_balance_change'].head(10))

3175441    310473.41
2283609   -669728.01
4834889    -18365.12
6159828         0.00
5962483   -116752.19
3833646    109220.34
2808571         0.00
3299225   -105990.86
4421469     -6706.04
529022      -6308.47
Name: dest_balance_change, dtype: float64


In [98]:
#Did the origin account go to zero after the transaction?

df['origin_zero'] = (df['newbalanceOrig'] == 0).astype(int)

In [99]:
df.columns


Index(['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest',
       'newbalanceDest', 'isFraud', 'type_CASH_OUT', 'type_DEBIT',
       'type_PAYMENT', 'type_TRANSFER', 'balance_change',
       'dest_balance_change', 'origin_zero'],
      dtype='object')

In [100]:
#Lets check to see if there are any missing values in the dataset

df.isnull().sum()

step                   0
amount                 0
oldbalanceOrg          0
newbalanceOrig         0
oldbalanceDest         0
newbalanceDest         0
isFraud                0
type_CASH_OUT          0
type_DEBIT             0
type_PAYMENT           0
type_TRANSFER          0
balance_change         0
dest_balance_change    0
origin_zero            0
dtype: int64

We see our dataset looks very clean now with no null values and the columns we will not be needing for our model dropped as well 

In [103]:
# Now let us save our transformed dataset to a new csv file

df.to_csv('transformed_data.csv', index=False)

print("Saved! Shape:", df.shape)

Saved! Shape: (180000, 14)
